In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys


In [11]:
from pathlib import Path

project_path = Path("/workspace")

skip_dirs = {
    ".Trash-0",
    ".ipynb_checkpoints",
    "__pycache__",
    ".git"
}

for path in sorted(project_path.rglob("*")):
    if not path.is_file():
        continue

    if any(part in skip_dirs for part in path.relative_to(project_path).parts):
        continue

    name = path.name.lower()

    if any(term in name for term in [
        "baseline",
        "benchmark",
        "features.py",
        "feature.py"
    ]):
        print(path)

/workspace/03_baseline.ipynb
/workspace/data/raw/feature.py
/workspace/src/feature.py
/workspace/src/features.py


In [12]:
from pathlib import Path
import ast

feature_file = Path("/workspace/src/features.py")

source_code = feature_file.read_text(encoding="utf-8")
tree = ast.parse(source_code)

print("=== FUNCTIONS AND CLASSES ===")

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(f"{type(node).__name__}: {node.name}")

print("\n=== COMPLETE FEATURES.PY SOURCE ===")
print(source_code)

=== FUNCTIONS AND CLASSES ===
FunctionDef: build_pipeline

=== COMPLETE FEATURES.PY SOURCE ===
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_pipeline(numeric_columns, categorical_columns):
    """
    Build reusable preprocessing for numeric and categorical features.
    The target column must not be included in either column list.
    """
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ], remainder="drop")
    return Pipeline([
 

In [13]:
file_path = "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

['README', 'data_dictionary', 'attrition_dataset', 'protected_attributes', 'attrition_scoring_2026', 'engagement_survey', 'performance_reviews', 'attendance_monthly_features', 'attendance_anomaly_labels', 'attendance_daily_examples', 'payroll_monthly_features', 'payroll_anomaly_labels', 'monthly_headcount_history', 'workforce_events', 'salary_training', 'job_descriptions', 'resumes', 'resume_job_labels', 'resume_bias_pairs', 'resume_protected_attrs', 'chatbot_nlu_training', 'chatbot_test_set', 'chatbot_responses', 'hr_faq', 'chatbot_dialogues', 'baseline_benchmarks', 'anomaly_baselines', 'sample_attrition_risk', 'sample_anomaly_results', 'sample_forecast', 'answer_key_planted_dirt', 'answer_key_notes']


In [14]:
benchmarks = pd.read_excel(file_path,sheet_name="baseline_benchmarks")

In [15]:
benchmarks

,model,roc_auc_mean,roc_auc_sd,pr_auc_mean,precision_mean,recall_mean,f1_mean,accuracy_mean,comment
0,Dummy (always stays),0.500,0.000,0.146,0.000,0.000,0.000,0.854,Looks 85% accurate yet catches nobody: why acc...
1,Logistic Regression (class_weight=balanced),0.713,0.010,0.332,0.260,0.630,0.368,0.683,Clean features only (reference target)
2,"Decision Tree (depth 4, balanced)",0.650,0.019,0.256,0.234,0.524,0.322,0.676,Clean features only (reference target)
3,"Random Forest (300 trees, balanced)",0.715,0.018,0.336,0.523,0.089,0.151,0.854,Clean features only (reference target)
4,LEAKY Logistic Regression (adds exit information),1.000,0.000,1.000,1.000,1.000,1.000,1.000,Too good to be true: this is data leakage


In [16]:
df_model = pd.read_csv(
    "/workspace/data/processed/attrition_features.csv"
)
target = "left_within_12m"

excluded_columns = [
    "row_id",
    "employee_id",
    "observation_date",
    "source",
    "voluntary_left_within_12m",
    "exit_date",
    "resignation_date",
    "exit_reason",
    "exit_interview_done",
    "fnf_settlement_amount"
]
X = df_model.drop(
    columns=excluded_columns + [target],
    errors="ignore"
)

y = df_model[target]


In [17]:
df_model.shape

(2389, 72)

In [18]:
 X.shape

(2389, 61)

In [19]:
 y.shape

(2389,)

In [20]:
y.value_counts(normalize=True).round(3)

left_within_12m
0    0.854
1    0.146
Name: proportion, dtype: float64

In [21]:
y.isna().sum()

np.int64(0)

In [22]:
print("\nExcluded leakage columns still in X:")
print([
    col for col in excluded_columns
    if col in X.columns
])


Excluded leakage columns still in X:
[]


In [ ]:
#baseline model 

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

In [31]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    f1_score,
    accuracy_score
)

In [35]:
sys.path.insert(0,"/workspace/src")

In [36]:
from features import build_pipeline

In [39]:
numeric_columns = X.select_dtypes(include="number").columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "string", "bool", "boolean"]
).columns.tolist()

In [ ]:
# Ensure no feature columns are silently overlooked

In [ ]:
unhandled_columns = [
    col for col in X.columns
    if col not in numeric_columns + categorical_columns
]
if unhandled_columns:
    raise ValueError(f"Review these feature columns: {unhandled_columns}")

In [ ]:
# Create five stratified folds with a fixed random state

In [40]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [42]:
folds = list(cv.split(X, y))

In [47]:
classifiers = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

In [48]:
# Train and evaluate using out-of-fold predictions

In [54]:
results = []
oof_probabilities = {}

for model_name, classifier in classifiers.items():

    model = Pipeline([
        ("preprocessing", build_pipeline(
            numeric_columns,
            categorical_columns
        )),
        ("classifier", classifier)
    ])

    probabilities = cross_val_predict(model,X,y,cv=folds,method="predict_proba")

    positive_index = list(np.sort(y.unique())).index(1)
    positive_probability = probabilities[:, positive_index]
    predictions = (positive_probability > 0.5).astype(int)

    oof_probabilities[model_name] = positive_probability

    results.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y, positive_probability),
        "pr_auc": average_precision_score(y, positive_probability),
        "precision": precision_score(
            y, predictions, zero_division=0
        ),
        "recall": recall_score(
            y, predictions, zero_division=0
        ),
        "f1": f1_score(
            y, predictions, zero_division=0
        ),
        "accuracy": accuracy_score(y, predictions)
    })

In [58]:
    

# Display the comparison table
results_df = pd.DataFrame(results).set_index("model")

results_df.round(3)



,roc_auc,pr_auc,precision,recall,f1,accuracy
model,,,,,,
Dummy,0.500,0.146,0.000,0.000,0.00,0.854
Logistic Regression,0.702,0.293,0.258,0.596,0.36,0.691


In [57]:
print("Cross-validation folds:", len(folds))
print("Random state: 42")
print("Feature count:", X.shape[1])
print("All rows have out-of-fold probabilities:",
      all(len(p) == len(y) for p in oof_probabilities.values()))

Cross-validation folds: 5
Random state: 42
Feature count: 61
All rows have out-of-fold probabilities: True


In [ ]:
# Comparison Note 

In [ ]:
# For the Dummy classifier the results are ROC-AUC=0.500 and Accuracy=0.854 but its recall and F1 score were zero beacuse it always predicted that employees would stay.
#logistic Regression gave result of ROC-AUC of 0.702 , which is 0.011 below the benchmark of 0.713.
#its PR-AUC of 0.293 and recall of 0.596 were below the benchmark values of 0.332 and 0.630 .Precision of 0.258 was very close to the benchmark of 0.683
#the ROC-AUC result does not trigger the task above 0.90 warning 